In [1]:
import cv2
import numpy as np
import os
from matplotlib import pyplot as plt
import time
import mediapipe as mp

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Flatten, BatchNormalization
from tensorflow.keras.layers import Input, TimeDistributed, Conv1D, MaxPooling1D
from tensorflow.keras.callbacks import TensorBoard, EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical

In [ ]:
import sounddevice as sd

def sine_beep(frequency=440, duration=1.0, volume=1.0):
    fs = 44100  # sample rate
    t = np.linspace(0, duration, int(fs*duration), False)
    tone = volume * np.sin(2 * np.pi * frequency * t)
    sd.play(tone, fs)
    sd.wait()

In [12]:
# Path for exported data, numpy arrays
DATA_PATH = os.path.join('MP_Data')

# Actions that we try to detect
actions = np.array(['yang', 'dan', 'dengan', 'ini', 'untuk'])

# Thirty videos worth of data
no_sequences = 30

# Videos are going to be 30 frames in length
sequence_length = 30

# Folder start
start_folder = 30

In [13]:
label_map = {label:num for num, label in enumerate(actions)}

log_dir = os.path.join('Logs')
tb_callback = TensorBoard(log_dir=log_dir)

sequences, labels = [], []
for action in actions:
    for sequence in np.array(os.listdir(os.path.join(DATA_PATH, action))).astype(int):
        window = []
        for frame_num in range(sequence_length):
            res = np.load(os.path.join(DATA_PATH, action, str(sequence), "{}.npy".format(frame_num)))
            window.append(res)
        sequences.append(window)
        labels.append(label_map[action])

In [14]:
print(np.shape(sequences))

(150, 30, 75, 3)


In [16]:
X = np.array(sequences)

y = to_categorical(labels).astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.05)

model = Sequential([

    Input(shape=(30, 75, 3)),  # (frame, landmark, xyz)

    # --------------------------
    # 1. CNN Spasial per frame
    # --------------------------
    TimeDistributed(Conv1D(64, kernel_size=3, activation='relu')),
    TimeDistributed(MaxPooling1D(pool_size=2)),
    TimeDistributed(BatchNormalization()),

    TimeDistributed(Conv1D(128, kernel_size=3, activation='relu')),
    TimeDistributed(MaxPooling1D(pool_size=2)),
    TimeDistributed(Flatten()),

    Dropout(0.3),

    # --------------------------
    # 2. LSTM Temporal
    # --------------------------
    LSTM(128, return_sequences=False),
    Dropout(0.3),

    # --------------------------
    # 3. Dense Layer
    # --------------------------
    Dense(256, activation='relu'),
    Dropout(0.3),

    Dense(128, activation='relu'),

    # Output
    Dense(actions.shape[0], activation='softmax')
])

model.compile(
    optimizer=Adam(learning_rate=0.0005),
    loss='categorical_crossentropy',
    metrics=['categorical_accuracy']
)

model.summary()

early_stop = EarlyStopping(monitor='val_loss', patience=15, min_delta=0.0005, restore_best_weights=True)

lr_scheduler = ReduceLROnPlateau(
    monitor='val_loss', factor=0.5, patience=5, min_delta=0.0001
)

model.fit(X_train, y_train, batch_size=16, epochs=200, callbacks=[tb_callback, early_stop, lr_scheduler])

# import tensorflowjs as tfjs
# tfjs.converters.save_keras_model(model, "model")

model.save('./my_model.keras')

sine_beep(1000, 10, 1)

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ time_distributed_24             │ (None, 30, 73, 64)     │           640 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_25             │ (None, 30, 36, 64)     │             0 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_26             │ (None, 30, 36, 64)     │           256 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_27             │ (None, 30, 34, 128)    │        24,704 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_28             │ (None, 30, 17, 128)    │             0 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_29             │ (None, 30, 2176)       │             0 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_12 (Dropout)            │ (None, 30, 2176)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_4 (LSTM)                   │ (None, 128)            │     1,180,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_13 (Dropout)            │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_12 (Dense)                │ (None, 256)            │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_14 (Dropout)            │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_14 (Dense)                │ (None, 5)              │           645 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,272,325 (4.85 MB)

 Trainable params: 1,272,197 (4.85 MB)

 Non-trainable params: 128 (512.00 B)

Epoch 1/200
9/9 ━━━━━━━━━━━━━━━━━━━━ 11s 138ms/step - categorical_accuracy: 0.3380 - loss: 1.5571 - learning_rate: 5.0000e-04
Epoch 2/200
1/9 ━━━━━━━━━━━━━━━━━━━━ 1s 156ms/step - categorical_accuracy: 0.8125 - loss: 1.1748

d:\Belajar koding\Realtime-Sign-Language-Detection-Using-LSTM-Model\myenv\Lib\site-packages\keras\src\callbacks\early_stopping.py:99: UserWarning: Early stopping conditioned on metric `val_loss` which is not available. Available metrics are: categorical_accuracy,loss
  current = self.get_monitor_value(logs)
d:\Belajar koding\Realtime-Sign-Language-Detection-Using-LSTM-Model\myenv\Lib\site-packages\keras\src\callbacks\callback_list.py:171: UserWarning: Learning rate reduction is conditioned on metric `val_loss` which is not available. Available metrics are: categorical_accuracy,loss,learning_rate.
  callback.on_epoch_end(epoch, logs)


9/9 ━━━━━━━━━━━━━━━━━━━━ 1s 140ms/step - categorical_accuracy: 0.5845 - loss: 1.1396 - learning_rate: 5.0000e-04
Epoch 3/200
9/9 ━━━━━━━━━━━━━━━━━━━━ 1s 136ms/step - categorical_accuracy: 0.5775 - loss: 0.9474 - learning_rate: 5.0000e-04
Epoch 4/200
9/9 ━━━━━━━━━━━━━━━━━━━━ 1s 139ms/step - categorical_accuracy: 0.7465 - loss: 0.7275 - learning_rate: 5.0000e-04
Epoch 5/200
9/9 ━━━━━━━━━━━━━━━━━━━━ 1s 139ms/step - categorical_accuracy: 0.8028 - loss: 0.5621 - learning_rate: 5.0000e-04
Epoch 6/200
9/9 ━━━━━━━━━━━━━━━━━━━━ 1s 138ms/step - categorical_accuracy: 0.8380 - loss: 0.4186 - learning_rate: 5.0000e-04
Epoch 7/200
9/9 ━━━━━━━━━━━━━━━━━━━━ 1s 137ms/step - categorical_accuracy: 0.9155 - loss: 0.3034 - learning_rate: 5.0000e-04
Epoch 8/200
9/9 ━━━━━━━━━━━━━━━━━━━━ 1s 136ms/step - categorical_accuracy: 0.9437 - loss: 0.2223 - learning_rate: 5.0000e-04
Epoch 9/200
9/9 ━━━━━━━━━━━━━━━━━━━━ 1s 140ms/step - categorical_accuracy: 0.9225 - loss: 0.2291 - learning_rate: 5.0000e-04
Epoch 10/200

# Evaluation using Confusion Matrix

In [17]:
from sklearn.metrics import multilabel_confusion_matrix, accuracy_score

yhat = model.predict(X_test)
ytrue = np.argmax(y_test, axis=1).tolist()
yhat = np.argmax(yhat, axis=1).tolist()
multilabel_confusion_matrix(ytrue, yhat)
accuracy_score(ytrue, yhat)

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step


1.0